In [1]:
import os
import os.path
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
import os
import sys
from pathlib import Path

# === CONFIGURATION ===
# Choose which dataset to run on: "val" or "test"
DATASET_MODE = "test"  # Change to "test" for final submission

# Set to True to rebuild indices from CSV (required on first run)
# Set to False to load cached indices (faster for subsequent runs)
FORCE_REBUILD_INDICES = False

# Detect environment
KAGGLE_ENV = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ENV:
    # Kaggle paths
    DATA_PATH = Path("/kaggle/input/omnilex-data")
    MODEL_PATH = Path("/kaggle/input/llama-model")
    OUTPUT_PATH = Path("/kaggle/working")
    INDEX_PATH = Path("/kaggle/input/omnilex-indices")
    sys.path.insert(0, "/kaggle/input/omnilex-utils")
else:
    # Local development paths
    REPO_ROOT = Path(".").resolve().parent
    DATA_PATH = REPO_ROOT / "data"
    MODEL_PATH = REPO_ROOT / "models"
    OUTPUT_PATH = REPO_ROOT / "output"
    INDEX_PATH = REPO_ROOT / "data" / "processed"

# CSV corpus files for index building
LAWS_CSV = DATA_PATH / "laws_de.csv"
COURTS_CSV = DATA_PATH / "court_considerations.csv"

# Index cache paths
LAWS_INDEX_PATH = INDEX_PATH / "laws_index.pkl"
COURTS_INDEX_PATH = INDEX_PATH / "courts_index.pkl"

# Derived paths based on DATASET_MODE
QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"
IS_VALIDATION_MODE = DATASET_MODE == "val"

# Create output directory
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
INDEX_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if KAGGLE_ENV else 'Local'}")
print(f"Dataset mode: {DATASET_MODE}")
print(f"Query file: {QUERY_FILE}")
print(f"Validation mode: {IS_VALIDATION_MODE}")
print(f"Force rebuild indices: {FORCE_REBUILD_INDICES}")
print(f"\nCorpus files:")
print(f"  Laws CSV: {LAWS_CSV} ({LAWS_CSV.stat().st_size / 1e6:.1f} MB)" if LAWS_CSV.exists() else f"  Laws CSV: {LAWS_CSV} (NOT FOUND)")
print(f"  Courts CSV: {COURTS_CSV} ({COURTS_CSV.stat().st_size / 1e9:.2f} GB)" if COURTS_CSV.exists() else f"  Courts CSV: {COURTS_CSV} (NOT FOUND)")
print(f"\nIndex cache: {INDEX_PATH}")

Environment: Local
Dataset mode: test
Query file: /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/test.csv
Validation mode: False
Force rebuild indices: False

Corpus files:
  Laws CSV: /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/laws_de.csv (73.0 MB)
  Courts CSV: /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/court_considerations.csv (2.43 GB)

Index cache: /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed


# 2. Load Corpora and Build/Load Indices

In [3]:
from bm25index import BM25Index
import bm25index

In [4]:
# Load or build laws index
# Laws CSV: ~45MB, ~269K rows
# Build time: ~30 seconds | Load from cache: <1 second

laws_index = bm25index.get_or_build_index(
    name="laws",
    csv_path=LAWS_CSV,
    index_path=LAWS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD_INDICES,
    max_rows=100000000  # Uncomment to test with smaller corpus
)
print(f"\nLaws index: {len(laws_index.documents):,} documents")

# Test search
test_results = laws_index.search("Vertrag", top_k=3)
print(f"\nTest search 'Vertrag': {len(test_results)} results")
if test_results:
    print(test_results)


Building laws index from /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/laws_de.csv
Counting rows in laws_de.csv...
Total rows to load: 272,433


Loading laws_de.csv:   0%|          | 0/272433 [00:00<?, ?it/s]


Building BM25 index for 175,933 documents...


split_compound: 100%|██████████| 175933/175933 [02:05<00:00, 1399.87it/s]


Split strings:   0%|          | 0/175933 [00:00<?, ?it/s]

...

Index built successfully!
Saving index to /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed/laws_index.pkl...
Index cached.

Laws index: 175,933 documents


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Test search 'Vertrag': 1 results
[{'query': 'Vertrag', 'hits': [{'rank': 1, 'score': 2.8019, 'text': '4 Wird ein Rahmen Vertrag mit nur einer Anbieterin abgeschlossen, so Wer Den die auf Die Sem Rahmen Vertrag beruhenden Einzel Verträge entsprechend den Bedingungen des Rahmen Vertrags abgeschlossen. Für den Abschluss der Einzel Verträge kann die Auftraggeberin die jeweilige Vertrags Partnerin schriftlich auffordern, ihr Ange Bot zu vervollständigen.', 'citation': 'Art. 25 Abs. 4 BöB'}, {'rank': 2, 'score': 2.783, 'text': '6 Ist das BAKOM Registerbetreiberin, so Unter Steht der Vertrag dem öffentlichen Recht (verwaltungsrechtlicher Vertrag); ist die Auf Gabe an einen Dritten übertragen, so Unter Steht der Vertrag dem Privat Recht (privatrechtlicher Vertrag).', 'citation': 'Art. 17 Abs. 6 VID'}, {'rank': 3, 'score': 2.7276, 'text': '1 Durch Vertrag kann die Verpflichtung zum Abschluss eines künftigen Vertrages begründet werden.', 'citation': 'Art. 22 Abs. 1 OR'}]}]


In [7]:
# Load or build courts index
# Courts CSV: ~2.3GB, ~2.5M rows
# Full corpus build time: ~15-20 minutes | Load from cache: ~10 seconds
# Full corpus can have peak memory during build: ~8-16GB

courts_index = bm25index.get_or_build_index(
    name="courts",
    csv_path=COURTS_CSV,
    index_path=COURTS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD_INDICES,
    max_rows=None,  # Change to use bigger corpus
    chunk_size = None,
    overlap_size = 100,
)
print(f"\nCourts index: {len(courts_index.documents):,} documents")

# Test search
test_results = courts_index.search("Meinungsfreiheit", top_k=1)
print(f"\nTest search 'Meinungsfreiheit': {len(test_results)} results")
if test_results:
    print(test_results)

Loading cached courts index from /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed/courts_index.pkl
  Loaded 2,476,315 documents

Courts index: 2,476,315 documents


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Test search 'Meinungsfreiheit': 1 results
[{'query': 'Meinungs Freiheit', 'hits': [{'rank': 1, 'score': 7.2207, 'text': '6.2.1. Die Meinungs Freiheit wird durch Art. 16 BV und Art. 10 EMRK garantiert. Gemäss Art. 16 Abs. 2 BV hat jede Per Son das Recht, ihre Meinung frei zu bilden, zu äussern und zu verbreiten. Gemäss Art. 10 Abs. 1 EMRK umfasst das Recht auf freie Meinungsäusserung die Meinungs Freiheit und die Freiheit, Informationen und Ideen ohne behördliche Ein Griffe und ohne Rück Sicht auf Staats Grenzen zu empfangen oder weiterzugeben. Dar Unter fallen die verschiedensten Formen der Kundgabe von Mein Ungen (BGE 143 I 147 E. 3.1; Urteil 6B_112/2025 vom 21. August 2025 E. 4.1.1, zur Publikation vorgesehen).', 'citation': '6B_1173/2023 E. 6.2.1'}]}]


In [6]:
# import pickle
# import re
# with open("../data/processed/courts_index.pkl.doc", "rb") as inf:
#     docs = pickle.load(inf)['documents']
#     for doc in docs:
#         print(doc['text'], len(doc['text'].split()), doc['citation'])